# Lab: Multi-Region Inference & KV Cache Locality

This lab explores the fundamental tradeoffs in multi-region LLM serving:
when to transfer KV cache across regions vs recompute locally, how routing
strategies affect time-to-first-token (TTFT), and how prefix pools amortize
costs across shared system prompts.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import List, Dict, Tuple

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

# --- Model & Network Parameters ---
NUM_LAYERS = 32
NUM_KV_HEADS = 8
HEAD_DIM = 128
BYTES_PER_PARAM = 2  # FP16

# KV cache size per token = 2 (K+V) * layers * kv_heads * head_dim * dtype
KV_BYTES_PER_TOKEN = 2 * NUM_LAYERS * NUM_KV_HEADS * HEAD_DIM * BYTES_PER_PARAM
print(f'KV cache per token: {KV_BYTES_PER_TOKEN / 1024:.1f} KB')
print(f'KV cache for 4K context: {KV_BYTES_PER_TOKEN * 4096 / 1024**2:.1f} MB')
print(f'KV cache for 128K context: {KV_BYTES_PER_TOKEN * 128_000 / 1024**3:.2f} GB')

## Transfer vs Recompute: The Crossover Point

At short contexts, recomputing prefill locally is cheaper than transferring
KV cache over the network. As context grows, transfer becomes attractive
because prefill cost scales quadratically (attention) while transfer scales linearly.

In [ ]:
# Network bandwidths (bytes/sec) between regions
INTER_REGION_BW = 10e9 / 8  # 10 Gbps typical cross-region
NETWORK_OVERHEAD_MS = 5.0     # TCP handshake + routing

# Prefill compute: approximate as O(n * d_model) per layer for linear,
# plus O(n^2 * head_dim) for attention. Use empirical fit from A100.
PREFILL_TFLOPS = 312e12  # A100 FP16 peak
PREFILL_EFFICIENCY = 0.4  # typical MFU for prefill
D_MODEL = NUM_KV_HEADS * HEAD_DIM * (32 // NUM_KV_HEADS)  # GQA ratio

def prefill_time_ms(seq_len: int) -> float:
    """Empirical prefill time approximation (ms) on A100."""
    # Quadratic component from attention + linear from FFN
    flops = NUM_LAYERS * (2 * seq_len * D_MODEL + 2 * seq_len**2 * HEAD_DIM)
    return (flops / (PREFILL_TFLOPS * PREFILL_EFFICIENCY)) * 1000

def transfer_time_ms(seq_len: int) -> float:
    """Time to transfer KV cache for seq_len tokens across regions."""
    kv_bytes = seq_len * KV_BYTES_PER_TOKEN
    return NETWORK_OVERHEAD_MS + (kv_bytes / INTER_REGION_BW) * 1000

# Sweep context lengths
ctx_lengths = np.array([512, 1024, 2048, 4096, 8192, 16384, 32768, 65536, 128000])
prefill_times = np.array([prefill_time_ms(n) for n in ctx_lengths])
transfer_times = np.array([transfer_time_ms(n) for n in ctx_lengths])

# Find crossover
crossover_idx = np.argmax(prefill_times > transfer_times)
crossover_len = ctx_lengths[crossover_idx]

print(f"{'Context Len':>12} {'Prefill (ms)':>14} {'Transfer (ms)':>14} {'Winner':>10}")
print('-' * 54)
for i, n in enumerate(ctx_lengths):
    winner = 'Transfer' if prefill_times[i] > transfer_times[i] else 'Recompute'
    print(f'{n:>12,} {prefill_times[i]:>14.2f} {transfer_times[i]:>14.2f} {winner:>10}')
print(f'\nCrossover point: ~{crossover_len:,} tokens')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(ctx_lengths, prefill_times, 'b-o', label='Local Prefill (recompute)', linewidth=2)
ax.plot(ctx_lengths, transfer_times, 'r-s', label='Cross-Region Transfer', linewidth=2)

# Annotate crossover
ax.axvline(x=crossover_len, color='gray', linestyle='--', alpha=0.7)
ax.annotate(f'Crossover\n~{crossover_len:,} tokens',
            xy=(crossover_len, prefill_times[crossover_idx]),
            xytext=(crossover_len * 2, prefill_times[crossover_idx] * 0.7),
            fontsize=11, ha='center',
            arrowprops=dict(arrowstyle='->', color='black'))

ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.set_xlabel('Context Length (tokens)', fontsize=12)
ax.set_ylabel('Time (ms)', fontsize=12)
ax.set_title('Transfer vs Recompute: When to Ship KV Cache', fontsize=14)
ax.legend(fontsize=11)
ax.set_xticks(ctx_lengths)
ax.set_xticklabels([f'{n//1024}K' if n >= 1024 else str(n) for n in ctx_lengths], rotation=45)
plt.tight_layout()
plt.savefig('transfer_vs_recompute.png', dpi=150, bbox_inches='tight')
plt.show()

## Multi-Region Routing Simulation

We simulate 10K inference requests across 3 regions and compare routing strategies:
- **Strategy A**: Always route to nearest region (recompute prefill locally)
- **Strategy B**: Transfer KV if context > crossover point
- **Strategy C**: Global prefix pool (shared system prompts pre-cached everywhere)

In [ ]:
@dataclass
class Region:
    name: str
    latency_to: Dict[str, float]  # ms one-way latency to other regions

regions = {
    'US': Region('US', {'US': 1, 'EU': 75, 'Asia': 120}),
    'EU': Region('EU', {'US': 75, 'EU': 1, 'Asia': 90}),
    'Asia': Region('Asia', {'US': 120, 'EU': 90, 'Asia': 1}),
}

# Generate 10K requests with realistic distribution
N_REQUESTS = 10_000
# User regions (60% US, 25% EU, 15% Asia)
user_regions = np.random.choice(['US', 'EU', 'Asia'], N_REQUESTS, p=[0.6, 0.25, 0.15])
# Context lengths: log-normal distribution (most short, some very long)
context_lens = np.clip(np.random.lognormal(mean=8, sigma=1.2, size=N_REQUESTS).astype(int), 256, 128000)
# 30% of requests share one of 5 system prompts (2K tokens each)
has_shared_prefix = np.random.random(N_REQUESTS) < 0.3
SHARED_PREFIX_LEN = 2048

# Region where KV cache currently lives (previous turn in conversation)
kv_regions = np.random.choice(['US', 'EU', 'Asia'], N_REQUESTS, p=[0.5, 0.3, 0.2])

DECODE_LATENCY_MS = 15  # fixed decode overhead

def simulate_strategy_a():
    """Always route nearest: recompute prefill locally."""
    ttfts = []
    for i in range(N_REQUESTS):
        ttft = prefill_time_ms(context_lens[i]) + DECODE_LATENCY_MS
        ttfts.append(ttft)
    return np.array(ttfts)

def simulate_strategy_b():
    """Transfer if context > crossover, else recompute."""
    ttfts = []
    for i in range(N_REQUESTS):
        if context_lens[i] > crossover_len and kv_regions[i] != user_regions[i]:
            # Transfer KV from origin region
            network_lat = regions[user_regions[i]].latency_to[kv_regions[i]]
            ttft = transfer_time_ms(context_lens[i]) + network_lat + DECODE_LATENCY_MS
        else:
            ttft = prefill_time_ms(context_lens[i]) + DECODE_LATENCY_MS
        ttfts.append(ttft)
    return np.array(ttfts)

def simulate_strategy_c():
    """Global prefix pool: shared prompts pre-cached, rest uses strategy B."""
    ttfts = []
    for i in range(N_REQUESTS):
        if has_shared_prefix[i]:
            # Only need to prefill the non-shared portion
            unique_len = max(256, context_lens[i] - SHARED_PREFIX_LEN)
            ttft = prefill_time_ms(unique_len) + DECODE_LATENCY_MS
        elif context_lens[i] > crossover_len and kv_regions[i] != user_regions[i]:
            network_lat = regions[user_regions[i]].latency_to[kv_regions[i]]
            ttft = transfer_time_ms(context_lens[i]) + network_lat + DECODE_LATENCY_MS
        else:
            ttft = prefill_time_ms(context_lens[i]) + DECODE_LATENCY_MS
        ttfts.append(ttft)
    return np.array(ttfts)

ttft_a = simulate_strategy_a()
ttft_b = simulate_strategy_b()
ttft_c = simulate_strategy_c()

# Cost model: $0.09/GB transferred, $0.01/GPU-sec compute
COST_PER_GB = 0.09
COST_PER_GPU_SEC = 0.01

def compute_cost(strategy: str) -> float:
    total = 0.0
    for i in range(N_REQUESTS):
        if strategy == 'A':
            total += (prefill_time_ms(context_lens[i]) / 1000) * COST_PER_GPU_SEC
        elif strategy == 'B':
            if context_lens[i] > crossover_len and kv_regions[i] != user_regions[i]:
                total += (context_lens[i] * KV_BYTES_PER_TOKEN / 1e9) * COST_PER_GB
            else:
                total += (prefill_time_ms(context_lens[i]) / 1000) * COST_PER_GPU_SEC
        else:  # C
            if has_shared_prefix[i]:
                unique_len = max(256, context_lens[i] - SHARED_PREFIX_LEN)
                total += (prefill_time_ms(unique_len) / 1000) * COST_PER_GPU_SEC
            elif context_lens[i] > crossover_len and kv_regions[i] != user_regions[i]:
                total += (context_lens[i] * KV_BYTES_PER_TOKEN / 1e9) * COST_PER_GB
            else:
                total += (prefill_time_ms(context_lens[i]) / 1000) * COST_PER_GPU_SEC
    return total

costs = [compute_cost('A'), compute_cost('B'), compute_cost('C')]

print(f"{'Strategy':<25} {'P50 TTFT (ms)':>14} {'P99 TTFT (ms)':>14} {'Cost ($)':>10}")
print('-' * 67)
for name, ttft, cost in zip(['A: Always Nearest', 'B: Transfer if Long', 'C: Global Prefix Pool'],
                             [ttft_a, ttft_b, ttft_c], costs):
    print(f'{name:<25} {np.percentile(ttft, 50):>14.1f} {np.percentile(ttft, 99):>14.1f} {cost:>10.2f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

strategies = ['A: Nearest', 'B: Transfer\nif Long', 'C: Prefix\nPool']
p50s = [np.percentile(t, 50) for t in [ttft_a, ttft_b, ttft_c]]
p99s = [np.percentile(t, 99) for t in [ttft_a, ttft_b, ttft_c]]

x = np.arange(3)
bars1 = ax1.bar(x - 0.2, p50s, 0.35, label='P50 TTFT', color='#2563eb', alpha=0.8)
bars2 = ax1.bar(x + 0.2, p99s, 0.35, label='P99 TTFT', color='#dc2626', alpha=0.8)
ax1.set_xticks(x)
ax1.set_xticklabels(strategies)
ax1.set_ylabel('TTFT (ms)')
ax1.set_title('Time-to-First-Token by Strategy')
ax1.legend()
ax1.bar_label(bars1, fmt='%.0f', fontsize=9)
ax1.bar_label(bars2, fmt='%.0f', fontsize=9)

bars3 = ax2.bar(x, costs, color=['#2563eb', '#16a34a', '#9333ea'], alpha=0.8)
ax2.set_xticks(x)
ax2.set_xticklabels(strategies)
ax2.set_ylabel('Cost ($) per 10K requests')
ax2.set_title('Cost Comparison')
ax2.bar_label(bars3, fmt='$%.2f', fontsize=10)

plt.tight_layout()
plt.savefig('routing_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Prefix Pool Savings

System prompts are shared across many requests. Replicating the KV cache of the
top-N system prompts to all regions eliminates redundant prefill computation.
Here we quantify the ROI.

In [ ]:
# Top system prompts and their usage frequency
TOP_PROMPTS = [
    {'name': 'Chat Assistant', 'tokens': 2048, 'daily_requests': 50_000},
    {'name': 'Code Copilot', 'tokens': 4096, 'daily_requests': 30_000},
    {'name': 'RAG QA', 'tokens': 1536, 'daily_requests': 20_000},
    {'name': 'Summarizer', 'tokens': 1024, 'daily_requests': 15_000},
    {'name': 'Agent Router', 'tokens': 3072, 'daily_requests': 10_000},
]

N_REGIONS = 3
STORAGE_COST_PER_GB_MONTH = 0.023  # S3 standard
GPU_COST_PER_HOUR = 2.0  # A100 on-demand approx

print(f"{'Prompt':<16} {'Tokens':>7} {'KV Size (MB)':>12} {'Storage $/mo':>13} "
      f"{'Prefill Saved':>14} {'Daily ROI':>10}")
print('-' * 78)

total_storage_cost = 0
total_savings = 0

for p in TOP_PROMPTS:
    kv_size_mb = p['tokens'] * KV_BYTES_PER_TOKEN / 1024**2
    # Storage cost: replicate across all regions
    storage_gb = (kv_size_mb * N_REGIONS) / 1024
    storage_monthly = storage_gb * STORAGE_COST_PER_GB_MONTH
    storage_daily = storage_monthly / 30
    
    # Prefill savings: time saved * requests * GPU cost
    prefill_saved_sec = prefill_time_ms(p['tokens']) / 1000 * p['daily_requests']
    savings_daily = (prefill_saved_sec / 3600) * GPU_COST_PER_HOUR
    
    roi = savings_daily / storage_daily if storage_daily > 0 else float('inf')
    
    total_storage_cost += storage_daily
    total_savings += savings_daily
    
    print(f"{p['name']:<16} {p['tokens']:>7,} {kv_size_mb:>12.1f} {storage_monthly:>13.4f} "
          f"${savings_daily:>12.2f} {roi:>9.0f}x")

print(f'\nTotal daily storage cost: ${total_storage_cost:.4f}')
print(f'Total daily compute savings: ${total_savings:.2f}')
print(f'Overall ROI: {total_savings/total_storage_cost:.0f}x')

## KV Cache Transfer Cost Analysis

Comparing the dollar cost of moving KV cache between regions using different
transport mechanisms: RDMA (InfiniBand), TCP (standard), and S3 (store-and-forward).

In [ ]:
# Transport costs and characteristics
transports = {
    'RDMA (InfiniBand)': {'bw_gbps': 400, 'cost_per_gb': 0.02, 'latency_ms': 0.5},
    'TCP (cross-region)': {'bw_gbps': 10, 'cost_per_gb': 0.09, 'latency_ms': 75},
    'S3 (store+forward)': {'bw_gbps': 25, 'cost_per_gb': 0.04, 'latency_ms': 50},  # PUT+GET
}

ctx_sweep = [512, 2048, 8192, 32768, 65536, 128000]

print(f"{'Context':>9} {'KV Size':>9}", end='')
for t in transports:
    print(f" | {t:>20} cost   time", end='')
print()
print('-' * 110)

for ctx in ctx_sweep:
    kv_gb = ctx * KV_BYTES_PER_TOKEN / 1e9
    kv_mb = kv_gb * 1000
    print(f"{ctx:>8,} {kv_mb:>7.1f}MB", end='')
    for name, spec in transports.items():
        cost = kv_gb * spec['cost_per_gb']
        time_ms = spec['latency_ms'] + (kv_gb * 8 / spec['bw_gbps']) * 1000
        print(f" | {cost*1000:>10.3f}m$ {time_ms:>7.1f}ms", end='')
    print()

print('\nNote: RDMA only available within same AZ/cluster. Cross-region requires TCP or S3.')
print('At 128K context, KV transfer costs ~$0.001 via TCP but saves ~$0.01 in prefill compute.')

## Key Takeaways

1. **Crossover point exists**: Below ~4-8K tokens, local recompute is faster than network transfer. Above it, shipping KV cache wins.
2. **Routing strategy matters**: Hybrid approaches (Strategy B/C) reduce P99 TTFT by 30-60% vs always-recompute.
3. **Prefix pools are extremely high-ROI**: Replicating top-5 system prompts costs pennies but saves hundreds of dollars daily in prefill compute.
4. **Transport choice depends on topology**: RDMA for intra-cluster (disaggregated prefill/decode), TCP for cross-region live transfer, S3 for batch/async replication.
5. **Cost is negligible vs compute**: Even at 128K context, transfer cost is ~$0.001 while the prefill compute it replaces costs ~$0.01.